In [1]:
from astropy.time import Time
import astropy.units as u
import json
import pandas as pd

In [2]:
# Function to read the JSON settings file
def get_config():
  
  # Get settings' directory
  settings_dir = r"../config/settings.json"

  # Open JSON file
  with open(settings_dir, "r") as file:
      settings = json.load(file)
  return settings

# Function to import local classification database
def get_localdatabase():
  
  # Data directory
  data_dir = r"../data/body_classification.csv"
  
  # Read CSV file
  return pd.read_csv(data_dir, delimiter=',')

In [164]:
asteroids = get_localdatabase()
settings = get_config()

families = settings['body']['fields']['family'] or ['is_neo', 'is_mba', 'is_jupiter_trojan', 'is_centaur', 'is_tno', 'is_other']
states = settings['body']['fields']['state'] or ['is_provisional', 'is_numbered']
obj_unc = settings['body']['fields']['orbit_uncertainty'] or asteroids['orbit_uncertainty'].unique()
limit = settings['body']['fields']['limit'] or 300

filtered_asteroids = asteroids[
  asteroids[families].any(axis=1) &
  asteroids[states].any(axis=1) &
  asteroids['orbit_uncertainty'].isin(obj_unc)
]['designation']

if not filtered_asteroids.empty:
  filtered_asteroids = filtered_asteroids.sample(n=limit).to_list()

filtered_asteroids

['(773669) 2020 MB8',
 '(186272) 2001 YZ140',
 '(291276) 2006 BE98',
 '(24467) 2000 SS165',
 '(809366) 2019 GE75',
 '(22199) Klonios',
 '(389318) 2009 SV169',
 '(634593) 2011 YD96',
 '(496293) 2013 AL55',
 '(846722) 2020 JD4',
 '(375669) 2009 FA32',
 '(420753) 2013 AW131',
 '(238623) 2005 CL12',
 '(571228) 2007 EV167',
 '(743196) 2008 JH50',
 '(572408) 2008 GD168',
 '(652767) 2014 EL158',
 '(669755) 2013 BS98',
 '(489535) 2007 RS153',
 '(728959) 2010 VA167',
 '(437759) 2014 GG47',
 '(315951) 2008 TL144',
 '(231623) 2009 SR207',
 '(764411) 2013 AM193',
 '(231493) 2008 QT19',
 '(375241) 2008 GM11',
 '(598405) 2008 RH175',
 '(778762) 2010 VT280',
 '(647067) 2008 OW32',
 '(347134) 2010 MC113',
 '(299061) 2005 CE31',
 '(576497) 2012 TX4',
 '(723681) 2007 ET149',
 '(629259) 2001 DS113',
 '(22180) Paeon',
 '(454257) 2013 RH98',
 '(621894) 2011 PG21',
 '(352659) 2008 RE2',
 '(266662) 2008 UJ209',
 '(129135) 2005 AD21',
 '(673520) 2015 DO133',
 '(341881) 2008 GM94',
 '(88268) 2001 KK76',
 '(594

In [170]:
# Functions to standardize input data

# Helper function to search body names
def _search_bodies(fields):
  
  # Default direct options
  obj_type = fields['object_type'] or 'asteroid'
  
  if obj_type == 'asteroid':
    asteroids = get_localdatabase()
    
    families = fields['family'] or ['is_neo', 'is_mba', 'is_jupiter_trojan', 'is_centaur', 'is_tno', 'is_other']
    states = fields['state'] or ['is_provisional', 'is_numbered']
    obj_unc = fields['orbit_uncertainty'] or asteroids['orbit_uncertainty'].unique()
    limit = fields['limit'] or 300
    
    asteroids = asteroids[
      asteroids[families].any(axis=1) &
      asteroids[states].any(axis=1) &
      asteroids['orbit_uncertainty'].isin(obj_unc)
    ]['designation']
      
    if not asteroids.empty:
      asteroids = asteroids.sample(n=limit).to_list()
      
    return asteroids

# Function to standardize input data
def default(settings):
  
  # Default values
  if not settings['limit_magnitude']:
    settings['limit_magnitude'] = 16
  if not settings['exposition_time']:
    settings['exposition_time'] = 5
  if not settings['database']:
    settings['database'] = ["JPL", "MPC"]
  if not settings['observer']['code'] and not settings['observer']['coord']:
    settings['observer']['code'] = "geo"
  if not settings['epoch']:
    settings['epoch'] = {
      "range": {
        "start": str(Time.now()),
        "stop": str(Time.now() + 1 * u.day),
        "step": "1m",
        "number": 720
      }
    }
  if not settings['body']['id']:
    settings['body']['id'] = _search_bodies(settings['body']['fields'])
    
  return settings

In [171]:
settings = get_config()
print(settings)

{'body': {'id': None, 'fields': {'object_type': 'asteroid', 'critical_list_numbered_object': None, 'limit': 5, 'family': ['is_centaur', 'is_tno', 'is_jupiter_trojan'], 'orbit_uncertainty': None, 'state': ['is_numbered']}}, 'database': None, 'epoch': None, 'observer': {'code': 'G37', 'coord': None}, 'limit_magnitude': None, 'exposition_time': None, 'ADS_key': None}


In [172]:
settings = default(settings)
print(settings)

{'body': {'id': ['(651362) 2013 AT36', '(24835) 1995 SM55', '(633399) 2009 SU151', '(667019) 2010 XN118', '(729018) 2010 XX83'], 'fields': {'object_type': 'asteroid', 'critical_list_numbered_object': None, 'limit': 5, 'family': ['is_centaur', 'is_tno', 'is_jupiter_trojan'], 'orbit_uncertainty': None, 'state': ['is_numbered']}}, 'database': ['JPL', 'MPC'], 'epoch': {'range': {'start': '2026-07-06 20:29:38.355592', 'stop': '2026-07-07 20:29:38.356044', 'step': '1m', 'number': 720}}, 'observer': {'code': 'G37', 'coord': None}, 'limit_magnitude': 16, 'exposition_time': 5, 'ADS_key': None}
